In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Display settings
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [4]:
# Load all datasets
orders = pd.read_csv('../data/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/olist_order_items_dataset.csv')
customers = pd.read_csv('../data/olist_customers_dataset.csv')
sellers = pd.read_csv('../data/olist_sellers_dataset.csv')
products = pd.read_csv('../data/olist_products_dataset.csv')
payments = pd.read_csv('../data/olist_order_payments_dataset.csv')
reviews = pd.read_csv('../data/olist_order_reviews_dataset.csv')
geolocation = pd.read_csv('../data/olist_geolocation_dataset.csv')
category_translation = pd.read_csv('../data/product_category_name_translation.csv')

In [5]:
# Overview of all datasets
datasets = {
    'orders': orders,
    'order_items': order_items,
    'customers': customers,
    'sellers': sellers,
    'products': products,
    'payments': payments,
    'reviews': reviews,
    'geolocation': geolocation,
    'category_translation': category_translation
}

for name, df in datasets.items():
    print(f"{'='*40}")
    print(f"Table: {name}")
    print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
    print(f"Columns: {list(df.columns)}")
    print()

Table: orders
Shape: 99441 rows x 8 columns
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Table: order_items
Shape: 112650 rows x 7 columns
Columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Table: customers
Shape: 99441 rows x 5 columns
Columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Table: sellers
Shape: 3095 rows x 4 columns
Columns: ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

Table: products
Shape: 32951 rows x 9 columns
Columns: ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

Table: payments
Shape: 103886 rows x 5 columns
Column

In [6]:
# Check missing values in each table
for name, df in datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]  # only show columns that have missing values
    
    print(f"{'='*40}")
    print(f"Table: {name}")
    if len(missing) == 0:
        print("No missing values! ✅")
    else:
        print(missing)
    print()

Table: orders
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

Table: order_items
No missing values! ✅

Table: customers
No missing values! ✅

Table: sellers
No missing values! ✅

Table: products
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

Table: payments
No missing values! ✅

Table: reviews
review_comment_title      87656
review_comment_message    58247
dtype: int64

Table: geolocation
No missing values! ✅

Table: category_translation
No missing values! ✅



In [7]:
# Handle missing values

# 1. Products - fill missing category with 'unknown'
products['product_category_name'] = products['product_category_name'].fillna('unknown')

# 2. Products - fill missing numeric columns with median
numeric_cols = ['product_name_lenght', 'product_description_lenght', 
                'product_photos_qty', 'product_weight_g',
                'product_length_cm', 'product_height_cm', 'product_width_cm']

for col in numeric_cols:
    median_val = products[col].median()
    products[col] = products[col].fillna(median_val)

# 3. Verify fixes
print("Missing values in products after fix:")
print(products.isnull().sum())

Missing values in products after fix:
product_id                    0
product_category_name         0
product_name_lenght           0
product_description_lenght    0
product_photos_qty            0
product_weight_g              0
product_length_cm             0
product_height_cm             0
product_width_cm              0
dtype: int64


In [9]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [11]:
# Translate Portuguese category names to English

# Merge products with translation table
products = products.merge(
    category_translation,      # the translation lookup table
    on='product_category_name', # match on this column
    how='left'                  # keep all products even if no translation exists
)

# For 'unknown' categories, fill english name as 'unknown' too
products['product_category_name_english'] = products['product_category_name_english'].fillna('unknown')

# Check result
print(f"Total products: {len(products)}")
print(f"\nSample translations:")
print(products[['product_category_name', 'product_category_name_english']].drop_duplicates().head(10))

Total products: 32951

Sample translations:
   product_category_name product_category_name_english
0             perfumaria                     perfumery
1                  artes                           art
2          esporte_lazer                sports_leisure
3                  bebes                          baby
4  utilidades_domesticas                    housewares
5  instrumentos_musicais           musical_instruments
6             cool_stuff                    cool_stuff
7       moveis_decoracao               furniture_decor
8       eletrodomesticos               home_appliances
9             brinquedos                          toys


In [12]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares
